In [43]:
import torch
import torch.optim as optim
import torch.nn as nn
from torchvision import transforms, datasets
from torch.utils.data import DataLoader, random_split
from Models import CNN_1

In [44]:
batch_size_=32
num_epochs = 10

In [45]:
train_transform = transforms.Compose([
    transforms.Resize(255),
    transforms.CenterCrop(224),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomRotation(30),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

val_test_transform = transforms.Compose([
    transforms.Resize(255),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

In [46]:

full_dataset = datasets.ImageFolder("./DATASET/FULL")

total_size = len(full_dataset)
train_size = int(0.7 * total_size)
val_size = int(0.2 * total_size)
test_size = total_size - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(full_dataset, [train_size, val_size, test_size])

In [47]:
from torch.utils.data import Subset


class DatasetWithTransform(torch.utils.data.Dataset):
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform
    def __getitem__(self, idx):
        x, y = self.subset[idx]  # PIL Image
        if self.transform:
            x = self.transform(x)  # → Tensor
        return x, y
    def __len__(self):
        return len(self.subset)


In [48]:
train_loader = DataLoader(DatasetWithTransform(train_dataset, train_transform), batch_size=32, shuffle=True)
val_loader = DataLoader(DatasetWithTransform(val_dataset, val_test_transform), batch_size=32, shuffle=False)
test_loader = DataLoader(DatasetWithTransform(test_dataset, val_test_transform), batch_size=32, shuffle=False)

In [49]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model1 = CNN_1().to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(model1.parameters(), lr=0.001)

In [50]:

for epoch in range(num_epochs):
    model1.train()
    total_loss = 0
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.float().unsqueeze(1).to(device)

        outputs = model1(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Total loss: {total_loss:.4f}")


Epoch 1, Total loss: 252.6061
Epoch 2, Total loss: 223.8574
Epoch 3, Total loss: 209.0322
Epoch 4, Total loss: 201.9140
Epoch 5, Total loss: 189.2277
Epoch 6, Total loss: 184.1104
Epoch 7, Total loss: 175.2012
Epoch 8, Total loss: 171.1126
Epoch 9, Total loss: 167.2066
Epoch 10, Total loss: 163.9347


In [51]:
model1.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model1(images)
        preds = (outputs > 0.5).int().squeeze()

        correct += (preds == labels).sum().item()
        total += labels.size(0)

print("Validation Accuracy:", correct / total)

Validation Accuracy: 0.882951146560319


In [52]:
model1.eval()

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)

        outputs = model1(images)
        preds = (outputs > 0.5).int()

        print(preds)
        break

tensor([[1],
        [0],
        [1],
        [0],
        [1],
        [0],
        [0],
        [1],
        [1],
        [0],
        [0],
        [1],
        [0],
        [1],
        [1],
        [0],
        [0],
        [0],
        [0],
        [0],
        [1],
        [0],
        [0],
        [1],
        [0],
        [1],
        [0],
        [1],
        [1],
        [0],
        [0],
        [0]], device='cuda:0', dtype=torch.int32)
